# OASIS Alzheimer — 3-class training (Kaggle)

Subject-level, leak-free training skeleton for the 3-class task
(`no_dementia` / `very_mild` / `mild_moderate`). Mirrors `research.md`.

**Runtime:** Kaggle Notebook, GPU (T4/P100) ON. Dataset: `ninadaithal/imagesoasis`.

Pipeline: rebuild the subject-level split on Kaggle (same seed → same folds) →
central-slice filter → two-stage transfer learning → **subject-level** metrics.


## 0. GPU sanity check (run this first)

Run this cell alone before anything else. If it reports `CPU only`, stop: turn the
accelerator on (sidebar -> Accelerator -> GPU T4 x2), let the session restart, then
Run All. Training this on CPU takes hours instead of minutes.


In [ ]:
import torch

print("torch      :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
print("device     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
if not torch.cuda.is_available():
    print("\n>>> NO GPU. Sidebar -> Accelerator -> GPU T4 x2, accept the restart, then Run All.")
    print(">>> Also check your weekly GPU quota -- if it is exhausted Kaggle silently gives you CPU.")


## 1. Config

In [ ]:
import os, re, random, glob
from pathlib import Path
import numpy as np, pandas as pd

def resolve_data_root(preferred):
    """Return `preferred` if it has class folders; otherwise auto-find the OASIS
    `Data` dir anywhere under /kaggle/input (handles repo uploads that nest it
    under datasets/Data/, and differing dataset slugs)."""
    if Path(preferred).is_dir():
        return preferred
    for cand in glob.glob("/kaggle/input/**/Data", recursive=True):
        if (Path(cand) / "Non Demented").is_dir():
            return cand
    return preferred  # fall through -> scan() returns empty -> guard fires with a clear message

CFG = dict(
    # Public dataset default; auto-resolved below for repo uploads (e.g. topotisarkar/oasis-dataset).
    data_root   = resolve_data_root("/kaggle/input/imagesoasis/Data"),
    seed        = 42,
    test_frac   = 0.20,
    n_folds     = 5,
    run_folds   = [0],          # [0] = smoke-test one fold; set to [0,1,2,3,4] for full CV
    # Slice window. 128..132 (5 slices/subject) trains fast but gives ~1.1k images,
    # far too few for an 11M-param net -> it memorises subjects. 124..136 is ~2.6x
    # the data for ~2.6x the epoch time, and is the single biggest anti-overfit lever.
    central_lo  = 124,
    central_hi  = 136,
    backbone    = "resnet18",   # "resnet18" (recommended baseline) | "resnet50" | "efficientnet_b0"
    img_size    = 224,
    batch_size  = 32,
    stage1_epochs = 3,          # frozen backbone, train head only
    stage2_epochs = 12,         # partial fine-tune (see freeze_until)
    # Stage 2 only unfreezes the LAST blocks. Fine-tuning all 11.18M params on ~3k
    # images is what collapsed train loss to 0.16 while val F1 fell. Note that
    # resnet18's parameters are back-loaded -- layer4 alone is 75% of the net -- so
    # freezing early blocks saves less than it looks like it should. Real but modest.
    #   "head"   -> linear probe, 1.5k trainable (strongest regulariser; same as stage 1)
    #   "layer4" -> train layer4 + head,   8.40M / 11.18M (75%)   <- default
    #   "layer3" -> train layer3 onward,  10.49M / 11.18M (94%)
    #   None     -> unfreeze everything (the old, overfitting behaviour)
    freeze_until = "layer4",
    lr_head      = 1e-3,        # stage 1: head on a frozen backbone
    lr_head_ft   = 2e-4,        # stage 2: head keeps a higher LR than the backbone
    lr_backbone  = 2e-5,        # stage 2: unfrozen backbone blocks
    weight_decay = 1e-2,        # AdamW decoupled WD; 1e-4 was far too weak for this data size
    label_smoothing = 0.05,     # stops the net driving train logits to certainty
    dropout      = 0.5,
    patience     = 4,           # early stop on val macro-F1
    num_workers  = 2,
    use_sampler  = True,        # WeightedRandomSampler for class balance (image-appropriate; see step 6).
    use_amp      = True,        # mixed precision -> ~2x faster on T4/P100 GPUs.
    cache_split  = True,        # cache the scanned/split table so reruns skip the slow file scan.
)
CLASS_NAMES = ["no_dementia", "very_mild", "mild_moderate"]
print("data_root ->", CFG["data_root"])

def set_seed(s):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(CFG["seed"])

## 2. Rebuild the split on Kaggle

Deterministic and subject-grouped, so it reproduces the local folds exactly
(same seed). Moderate → Mild merge, central-slice filter applied here.


In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from PIL import Image

FOLDER_TO_LABEL = {           # raw folder -> 3-class label (Moderate folded into Mild)
    "Non Demented": 0, "Very mild Dementia": 1,
    "Mild Dementia": 2, "Moderate Dementia": 2,
}
SUBJECT_RE = re.compile(r"(OAS1_\d+)", re.IGNORECASE)
SLICE_RE   = re.compile(r"_(\d+)\.(?:jpg|jpeg|png)$", re.IGNORECASE)

def scan(data_root):
    rows = []
    for folder, label in FOLDER_TO_LABEL.items():
        d = Path(data_root) / folder
        for p in sorted(d.glob("*")):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}: continue
            sm, im = SUBJECT_RE.search(p.name), SLICE_RE.search(p.name)
            if not sm: continue
            rows.append(dict(filepath=str(p), subject_id=sm.group(1).upper(),
                             label=label, slice_idx=int(im.group(1)) if im else -1))
    return pd.DataFrame(rows)

def drop_unreadable(df):
    """Filter out images PIL can't decode (truncated/corrupt uploads). Reopens and
    fully loads each file, since verify() alone misses some truncation."""
    ok = []
    for p in df["filepath"]:
        try:
            with Image.open(p) as im:
                im.load()
            ok.append(True)
        except Exception:
            ok.append(False)
    ok = pd.Series(ok, index=df.index)
    bad = int((~ok).sum())
    if bad:
        print(f"WARNING: dropped {bad} unreadable image(s) of {len(df)} "
              f"({bad/len(df):.1%}). If this is most of the set, the Kaggle "
              f"upload is corrupt -> re-upload the dataset.")
    return df[ok].copy()

def assign_folds(slices, test_frac, n_folds, seed):
    subj = (slices.groupby("subject_id")["label"].first().reset_index()
            .sort_values("subject_id").reset_index(drop=True))
    y = subj["label"].to_numpy()
    dev_idx, _ = next(StratifiedShuffleSplit(1, test_size=test_frac,
                                             random_state=seed).split(subj, y))
    subj["fold"] = -1
    dev = subj.iloc[dev_idx].reset_index()
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=seed)
    for f, (_, val) in enumerate(skf.split(dev, dev["label"])):
        subj.loc[dev.loc[val, "index"].to_numpy(), "fold"] = f
    return slices.merge(subj[["subject_id", "fold"]], on="subject_id", how="left")

def build_central():
    """Scan files -> assign folds -> keep central slices -> drop unreadable ones."""
    slices = scan(CFG["data_root"])
    assert len(slices), (
        f"No images found under {CFG['data_root']!r}. Check the Add Input path / data_root. "
        f"Folders present under /kaggle/input: {glob.glob('/kaggle/input/*')}")
    slices = assign_folds(slices, CFG["test_frac"], CFG["n_folds"], CFG["seed"])
    central = slices[slices.slice_idx.between(CFG["central_lo"], CFG["central_hi"])].copy()
    central = drop_unreadable(central)          # verify only the slices we actually train on
    assert len(central), "All central-slice images were unreadable -> re-upload the dataset."
    return central

# Cache: scanning tens of thousands of files is slow, so save the split table and
# reuse it on later runs. The cache key includes the slice window, so changing
# central_lo/central_hi rebuilds automatically. Note: /kaggle/working starts empty
# each session unless you save the notebook output / enable persistence.
CACHE = Path(f"/kaggle/working/central_{CFG['central_lo']}_{CFG['central_hi']}.parquet")
if CFG["cache_split"] and CACHE.exists():
    central = pd.read_parquet(CACHE)
    print(f"loaded cached split from {CACHE.name}")
else:
    central = build_central()
    if CFG["cache_split"]:
        central.to_parquet(CACHE)
        print(f"saved split cache -> {CACHE.name}")

print(f"central {CFG['central_lo']}-{CFG['central_hi']} (readable): {len(central):,}")
print(central.groupby("fold")["subject_id"].nunique())

## 3. Dataset + transforms

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate end-truncated JPEGs

IMAGENET_MEAN, IMAGENET_STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def build_tf(train):
    """Train-time augmentation is deliberately aggressive: with only ~13 near-identical
    slices per subject, weak augmentation (the old 10-deg rotation + flip) lets the net
    memorise individuals. RandomResizedCrop changes the framing every epoch, the affine
    shift breaks position memorisation, and RandomErasing forces it to use more than one
    region of the brain. Val/test stay deterministic."""
    if not train:
        return transforms.Compose([
            transforms.Resize((CFG["img_size"], CFG["img_size"])),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return transforms.Compose([
        transforms.RandomResizedCrop(CFG["img_size"], scale=(0.75, 1.0), ratio=(0.9, 1.11)),
        transforms.RandomHorizontalFlip(0.5),          # brains are near-symmetric; safe here
        transforms.RandomAffine(degrees=12, translate=(0.06, 0.06)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
    ])

class OasisDS(Dataset):
    def __init__(self, df, train, retries=4):
        self.df = df.reset_index(drop=True); self.tf = build_tf(train)
        self.retries = retries

    def __len__(self): return len(self.df)

    def _load(self, i):
        r = self.df.iloc[i]
        img = Image.open(r.filepath).convert("L")           # grayscale
        img = Image.merge("RGB", (img, img, img))            # -> 3 channels
        return self.tf(img), int(r.label), r.subject_id

    def __getitem__(self, i):
        # Flaky GitHub-import mounts occasionally return partial reads; retry the
        # same file, then fall back to the next index so one bad read never
        # kills the epoch.
        for attempt in range(self.retries):
            try:
                return self._load((i + attempt) % len(self.df))
            except Exception as e:
                if attempt == self.retries - 1:
                    print(f"WARN: skipping unreadable {self.df.iloc[i].filepath} ({e})")
        return self._load((i + self.retries) % len(self.df))

## 4. Model factory (backbone-agnostic)

In [ ]:
import torch.nn as nn
from torchvision import models

HEAD_ATTR = {"resnet18": "fc", "resnet50": "fc", "efficientnet_b0": "classifier"}

# Blocks in order, shallow -> deep. Stage 2 unfreezes from `freeze_until` onward.
BLOCK_ORDER = {
    "resnet18": ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"],
    "resnet50": ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"],
    # efficientnet_b0 has one `features` Sequential; index it by stage.
    "efficientnet_b0": [f"features.{i}" for i in range(9)],
}

def build_model(backbone, n_classes=3, dropout=None):
    dropout = CFG["dropout"] if dropout is None else dropout
    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        feat = m.fc.in_features
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feat = m.fc.in_features
    elif backbone == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        feat = m.classifier[1].in_features
    else:
        raise ValueError(backbone)
    setattr(m, HEAD_ATTR[backbone], nn.Sequential(nn.Dropout(dropout), nn.Linear(feat, n_classes)))
    return m

def set_trainable(model, backbone, freeze_until):
    """Freeze everything shallower than `freeze_until`; train that block onward + head.

    freeze_until="head" -> backbone fully frozen (stage 1)
    freeze_until=None   -> nothing frozen (full fine-tune)
    freeze_until="layer3" -> conv1/bn1/layer1/layer2 frozen, layer3+layer4+head trained
    """
    head = HEAD_ATTR[backbone]
    order = BLOCK_ORDER[backbone]
    if freeze_until == "head":
        frozen = set(order)
    elif freeze_until is None:
        frozen = set()
    else:
        assert freeze_until in order, f"{freeze_until!r} not in {order}"
        frozen = set(order[:order.index(freeze_until)])

    for name, p in model.named_parameters():
        if name.startswith(head):
            p.requires_grad = True
        else:
            p.requires_grad = not any(name.startswith(b + ".") or name == b for b in frozen)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"  trainable: {n_tr/1e6:.2f}M / {n_all/1e6:.2f}M params (freeze_until={freeze_until})")

def freeze_frozen_bn(model):
    """Put BatchNorm layers whose params are frozen into eval mode.

    model.train() would otherwise keep updating their running mean/var even though
    requires_grad=False — the frozen ImageNet statistics drift toward this tiny
    dataset, which is a quiet source of overfitting during partial fine-tuning.
    """
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            if not any(p.requires_grad for p in mod.parameters(recurse=False)):
                mod.eval()

def param_groups(model, backbone, lr_backbone, lr_head):
    """Discriminative LRs: pretrained backbone blocks move slowly, the new head faster."""
    head = HEAD_ATTR[backbone]
    bb = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith(head)]
    hd = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith(head)]
    groups = []
    if bb: groups.append({"params": bb, "lr": lr_backbone})
    if hd: groups.append({"params": hd, "lr": lr_head})
    return groups

## 5. Train / eval helpers (subject-level metrics)

In [ ]:
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = CFG["use_amp"] and device == "cuda"   # mixed precision only helps (and is safe) on GPU

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    if train:
        model.train()
        freeze_frozen_bn(model)   # keep frozen blocks' BN running stats pinned
    else:
        model.eval()
    total = 0.0
    with torch.set_grad_enabled(train):
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            if train: optimizer.zero_grad()
            # autocast runs the forward pass in fast float16 where it's safe.
            with torch.autocast(device_type="cuda", enabled=AMP):
                out = model(x); loss = criterion(out, y)
            if train:
                if scaler is not None:
                    # GradScaler prevents float16 gradients from underflowing to zero.
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            total += loss.item() * x.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def subject_eval(model, loader, criterion=None):
    """Subject-level metrics. Pass `criterion` to also get val loss — watching
    train loss alone hides overfitting until the F1 has already collapsed."""
    model.eval()
    probs, subs, ys, loss_sum = [], [], [], 0.0
    for x, y, s in loader:
        x = x.to(device)
        logits = model(x)
        if criterion is not None:
            loss_sum += criterion(logits, y.to(device)).item() * x.size(0)
        probs.append(torch.softmax(logits, 1).cpu().numpy()); subs.extend(s); ys.extend(y.numpy())
    probs = np.concatenate(probs)
    df = pd.DataFrame(probs, columns=CLASS_NAMES); df["subject_id"] = subs; df["y"] = ys
    g = df.groupby("subject_id")
    pred = g[CLASS_NAMES].mean().values.argmax(1)      # mean-prob aggregation
    true = g["y"].first().values
    return dict(macro_f1=f1_score(true, pred, average="macro"),
                bal_acc=balanced_accuracy_score(true, pred),
                val_loss=(loss_sum / len(loader.dataset)) if criterion is not None else float("nan"),
                report=classification_report(true, pred, target_names=CLASS_NAMES, zero_division=0),
                cm=confusion_matrix(true, pred), true=true, pred=pred)

def class_weights(df):
    counts = df["label"].value_counts().sort_index().values
    w = counts.sum() / (len(counts) * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)

## 6. Two-stage training over the selected fold(s)

In [ ]:
from torch.utils.data import WeightedRandomSampler

def make_train_loader(tr):
    """Training loader. With use_sampler, draw rare-class images more often
    (WeightedRandomSampler) — the image-friendly way to fight class imbalance,
    instead of inventing fake images (SMOTE/ADASYN don't fit raw images)."""
    ds = OasisDS(tr, True)
    if CFG["use_sampler"]:
        cw = class_weights(tr).cpu().numpy()          # weight per class
        sample_w = cw[tr["label"].to_numpy()]         # weight per image
        sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
        return DataLoader(ds, CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
    return DataLoader(ds, CFG["batch_size"], shuffle=True,
                      num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)

def train_fold(fold):
    tr = central[(central.fold != fold) & (central.fold != -1)]
    va = central[central.fold == fold]
    print(f"\n=== fold {fold}: train {tr.subject_id.nunique()} subj / {len(tr)} img  "
          f"| val {va.subject_id.nunique()} subj / {len(va)} img ===")
    tr_dl = make_train_loader(tr)
    va_dl = DataLoader(OasisDS(va, False), CFG["batch_size"], shuffle=False,
                       num_workers=CFG["num_workers"], pin_memory=True)

    model = build_model(CFG["backbone"]).to(device)
    # If the sampler already balances classes, don't also weight the loss (would double-count).
    criterion = nn.CrossEntropyLoss(
        label_smoothing=CFG["label_smoothing"],
        weight=None if CFG["use_sampler"] else class_weights(tr))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)

    # Stage 1 — frozen backbone, head only.
    set_trainable(model, CFG["backbone"], "head")
    opt = torch.optim.AdamW(param_groups(model, CFG["backbone"], CFG["lr_backbone"], CFG["lr_head"]),
                            weight_decay=CFG["weight_decay"])
    for e in range(CFG["stage1_epochs"]):
        l = run_epoch(model, tr_dl, criterion, opt, scaler)
        print(f"  [s1 {e+1}] train_loss={l:.4f}")

    # Baseline: the stage-1 head is often already competitive. Seed `best` with it so
    # stage 2 can only ever be kept if it genuinely beats the frozen-backbone model.
    m0 = subject_eval(model, va_dl, criterion)
    print(f"  [s1 end] val_macroF1={m0['macro_f1']:.4f}  val_balAcc={m0['bal_acc']:.4f}")
    best = m0["macro_f1"]
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_tag, wait = "stage1", 0

    # Stage 2 — unfreeze only the deep blocks, discriminative LRs, cosine decay.
    set_trainable(model, CFG["backbone"], CFG["freeze_until"])
    opt = torch.optim.AdamW(param_groups(model, CFG["backbone"], CFG["lr_backbone"], CFG["lr_head_ft"]),
                            weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["stage2_epochs"])
    for e in range(CFG["stage2_epochs"]):
        l = run_epoch(model, tr_dl, criterion, opt, scaler)
        m = subject_eval(model, va_dl, criterion)
        sched.step()
        gap = m["val_loss"] - l          # widening gap = the net is memorising the train set
        print(f"  [s2 {e+1}] train_loss={l:.4f}  val_loss={m['val_loss']:.4f}  gap={gap:+.3f}  "
              f"val_macroF1={m['macro_f1']:.4f}  val_balAcc={m['bal_acc']:.4f}")
        if m["macro_f1"] > best:
            best = m["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_tag, wait = f"s2e{e+1}", 0
        else:
            wait += 1
            if wait >= CFG["patience"]: print("  early stop"); break

    model.load_state_dict(best_state)
    # Save the best weights so you can reload instead of retraining next session.
    ckpt = f"/kaggle/working/model_fold{fold}.pt"
    torch.save({"state_dict": best_state, "cfg": CFG, "fold": fold,
                "val_macro_f1": best, "best_epoch": best_tag,
                "class_names": CLASS_NAMES}, ckpt)
    print(f"  saved -> {ckpt}  (best val_macroF1={best:.4f} @ {best_tag})")
    return model, subject_eval(model, va_dl, criterion)

results = {}
for f in CFG["run_folds"]:
    _, results[f] = train_fold(f)

## 7. Results (subject-level, vs 76.7% majority baseline)

In [ ]:
f1s  = [results[f]["macro_f1"] for f in results]
accs = [results[f]["bal_acc"]  for f in results]
print(f"macro-F1  : {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")
print(f"bal-acc   : {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print("majority baseline accuracy = 0.767 (predict no_dementia for all)\n")
for f in results:
    print(f"--- fold {f} ---"); print(results[f]["report"])
    print("confusion matrix (rows=true, cols=pred):"); print(results[f]["cm"], "\n")

## 8. Persist outputs (so a dead session can't eat your checkpoints)

`/kaggle/working` is scratch on the running container. It is wiped when the session
ends **and when you change the accelerator** — that is what deleted `model_fold0.pt`.

Two things that always work, in priority order:

1. **Download the zip the cell below prints.** One click, no setup, no internet
   needed, nothing to forget. Do this every time a run finishes.
2. **Save Version → "Save & Run All (Commit)".** Kaggle re-runs the notebook
   top-to-bottom on a clean machine and whatever lands in `/kaggle/working`
   becomes a permanent Output — downloadable later, and attachable as an input
   dataset to another notebook. This is the closest thing to "automatic": you
   press it once and it persists the run without further babysitting.

A third option, **Settings → Persistence → "Files only"**, keeps `/kaggle/working`
across restarts of this notebook. If you see it in your sidebar, turn it on — but
it is not present in every Kaggle UI/notebook type, and Kaggle treats it as
best-effort (it can fail on crashes or large files). Treat it as a bonus, never
as your only copy.

**Do not rely on the accelerator dropdown being safe.** Set it once, before you
start, and leave it alone for the rest of the session.


In [ ]:
import json, zipfile
from pathlib import Path
from IPython.display import FileLink, display

WORK      = Path("/kaggle/working")
ARTIFACTS = ("*.pt", "*.parquet", "*.csv")      # what's worth keeping (outputs.zip excluded)


def persist_outputs(extra_meta=None):
    """Bundle every artifact in /kaggle/working and hand back a download link."""
    files = sorted({p for pat in ARTIFACTS for p in WORK.glob(pat)})
    if not files:
        print("nothing to persist -- /kaggle/working has no artifacts yet")
        return None

    print("artifacts in /kaggle/working:")
    for f in files:
        print(f"  {f.name:44s} {f.stat().st_size/1e6:8.2f} MB")

    # A small run-summary rides along so a downloaded zip is self-describing.
    meta = WORK / "run_summary.json"
    meta.write_text(json.dumps({
        "cfg": {k: str(v) for k, v in CFG.items()},
        "folds": list(results),
        "macro_f1": {str(k): float(results[k]["macro_f1"]) for k in results},
        "bal_acc":  {str(k): float(results[k]["bal_acc"])  for k in results},
        **(extra_meta or {}),
    }, indent=2))

    bundle = WORK / "outputs.zip"
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files + [meta]:
            z.write(f, f.name)
    print(f"\nbundled -> {bundle.name} ({bundle.stat().st_size/1e6:.2f} MB)")

    print("\n" + "=" * 72)
    print(">>> CLICK THE LINK BELOW NOW. /kaggle/working is erased when this")
    print(">>> session ends or the accelerator changes -- there is no undo.")
    print("=" * 72)
    display(FileLink(bundle.name))          # FileLink paths are relative to /kaggle/working
    print("\nThen, to keep a permanent copy on Kaggle itself:")
    print("  Save Version -> 'Save & Run All (Commit)'  -> becomes a downloadable Output.")
    return bundle


persist_outputs()

## Notes / next steps
- Set `run_folds = [0,1,2,3,4]` for full CV; report mean ± std (done above). **One fold
  here has 56 subjects and only 4 `mild_moderate` — a single subject flipping moves
  macro-F1 by ~0.05, so a single-fold number is mostly noise. Judge changes on 5 folds.**
- Only evaluate the **test set** (`fold == -1`) once, after the recipe is frozen.
- Try `backbone = "resnet50"` as a second model and compare macro-F1.
- Add Grad-CAM on a few subjects for a sanity check against hidden leakage (research.md §9b).

### resnet18 parameter budget (3-class head)
| block | params | share |
|---|---:|---:|
| conv1 + bn1 | 9,536 | 0.1% |
| layer1 | 147,968 | 1.3% |
| layer2 | 525,568 | 4.7% |
| layer3 | 2,099,712 | 18.8% |
| layer4 | 8,393,728 | **75.1%** |
| fc head | 1,539 | 0.0% |
| **total** | **11,178,051** | |

Capacity is concentrated in `layer4`, so `freeze_until` is a coarse dial: `"layer3"`
still trains 94% of the net, `"layer4"` trains 75%, `"head"` trains 0.01%.

### Overfitting knobs (in `CFG`, roughly most → least effective)
| Knob | Now | If val F1 still falls while train loss drops |
|---|---|---|
| `central_lo/hi` | 124–136 | widen to 118–142 (more real data beats every other fix) |
| augmentation | RRC + affine + erasing | already aggressive; add mixup before loosening |
| `weight_decay` | 1e-2 | 3e-2 |
| `freeze_until` | `"layer4"` | `"head"` — a linear probe on ~3k images is a serious baseline, not a cop-out |
| `lr_backbone` | 2e-5 | 1e-5 |
| `dropout` | 0.5 | 0.6 |
| `label_smoothing` | 0.05 | 0.1 |

Stage 2 can now only *replace* the stage-1 model if it beats it on val macro-F1, so if
fine-tuning is hopeless on this data size you automatically keep the linear probe.

If instead **train loss stays high and val F1 plateaus low**, you've over-corrected —
loosen the other way (`freeze_until="layer3"`, `weight_decay=1e-3`).

### Checkpoint format changed
`model_fold{k}.pt` is now a dict, not a bare state_dict. To reload:

```python
ck = torch.load("model_fold0.pt", map_location="cpu", weights_only=False)
model = build_model(ck["cfg"]["backbone"])
model.load_state_dict(ck["state_dict"])
print(ck["val_macro_f1"], ck["best_epoch"])
```
